# Fehlerbehandlung

Ein Programm, das bei der ersten unerwarteten Eingabe abstürzt, ist in der Praxis unbrauchbar: Ein Messprogramm soll nicht abbrechen, weil eine Zeile der Messdatei beschädigt ist, und ein Berechnungstool soll bei einem Tippfehler eine verständliche Meldung ausgeben statt eines Tracebacks.

In diesem Anhang lernen Sie,

- welche Arten von Fehlern es gibt und wie man eine Fehlermeldung liest,
- wie man Laufzeitfehler mit `try` und `except` abfängt,
- wie man in eigenen Funktionen selbst Fehler mit `raise` auslöst und
- welche typischen Fehler man bei der Fehlerbehandlung vermeiden sollte.

## Arten von Fehlern

1. **Syntaxfehler** (`SyntaxError`, `IndentationError`): Der Code ist kein gültiges Python, z.B. weil eine Klammer oder ein Doppelpunkt fehlt. Er läuft gar nicht erst los und muss korrigiert werden.
2. **Laufzeitfehler** (*Exceptions*, „Ausnahmen“): Der Code ist korrekt, aber bei der Ausführung tritt ein Problem auf, etwa eine Division durch null oder eine fehlende Datei. Um diese Fehler geht es in diesem Anhang.
3. **Logische Fehler**: Das Programm läuft durch, liefert aber ein falsches Ergebnis, z.B. weil Grad und Bogenmaß verwechselt wurden. Hier helfen nur Tests und Plausibilitätsprüfungen.

## Eine Fehlermeldung lesen

Wenn eine Exception auftritt und nicht behandelt wird, bricht Python ab und gibt einen **Traceback** aus. Ein Beispiel: Ein Messwert wurde mit deutschem Dezimalkomma eingegeben.

In [1]:
def mean_value(values_as_text):
    numbers = [float(v) for v in values_as_text]
    return sum(numbers) / len(numbers)

mean_value(["21.4", "22,8", "23.1"])  # FEHLER: "22,8" ist keine gültige Zahl

ValueError: could not convert string to float: '22,8'

Ein Traceback wird am besten **von unten nach oben** gelesen: Die **letzte Zeile** nennt den Fehlertyp (`ValueError`) und eine Beschreibung – damit ist meist schon klar, was passiert ist. Darüber steht der **Weg** zur Fehlerstelle, hier vom Aufruf `mean_value(...)` bis zur Zeile mit `float(v)`. Die Fehlerstelle liegt oft tief in einer Funktion, die eigentliche Ursache aber beim Aufruf (hier: falsche Daten).

Die wichtigsten eingebauten Exceptions:

| Exception | Typische Ursache | Beispiel |
|---|---|---|
| `ValueError` | richtiger Typ, aber unbrauchbarer Wert | `float("22,8")`, `int("3.5")` |
| `TypeError` | Operation passt nicht zum Typ | `"Wert: " + 5`, `len(42)` |
| `ZeroDivisionError` | Division durch null | `1 / 0` |
| `IndexError` | Index außerhalb einer Sequenz | `[1, 2, 3][5]` |
| `KeyError` | Schlüssel nicht im Dictionary | `{"a": 1}["b"]` |
| `NameError` | Name nicht definiert (oft ein Tippfehler) | `pirnt("Hallo")` |
| `AttributeError` | Objekt hat das Attribut/die Methode nicht | `[1, 2].add(3)` |
| `FileNotFoundError` | Datei existiert nicht | `open("gibtsnicht.txt")` |

## Fehler abfangen mit `try` und `except`

Code, bei dem ein Fehler zu erwarten ist, schreibt man in einen `try`-Block. Tritt darin eine Exception auf, springt Python sofort in den passenden `except`-Block, statt abzubrechen:

In [2]:
text = "22,8"

try:
    value = float(text)
    print("Umgewandelt:", value)  # wird übersprungen, wenn float() scheitert
except ValueError:
    print(f"'{text}' ist keine gültige Zahl.")

print("Das Programm läuft weiter.")

'22,8' ist keine gültige Zahl.
Das Programm läuft weiter.


Ablauf im Detail:

- Python führt den `try`-Block Zeile für Zeile aus.
- Tritt **kein** Fehler auf, wird der `except`-Block übersprungen.
- Tritt ein Fehler auf, wird der Rest des `try`-Blocks übersprungen. Passt der Fehlertyp zu `except`, wird dieser Block ausgeführt und das Programm danach normal fortgesetzt. Passt er nicht, bricht das Programm wie gewohnt ab.

### Mehrere Fehlertypen und die Fehlermeldung selbst {.unnumbered}

Ein `try`-Block kann mehrere `except`-Blöcke haben, die nacheinander geprüft werden. Mehrere Fehlertypen lassen sich auch als Tupel zusammenfassen, z.B. `except (ValueError, TypeError) as e:`. Mit `as e` erhalten Sie das Fehlerobjekt selbst und können z.B. die Originalmeldung ausgeben:

In [4]:
def resistance(voltage_V, current_A):
    return voltage_V / current_A

measurements = [(12.0, 0.5), (5.0, 0.0), ("230", 2.0)]

for u, i in measurements:
    try:
        r = resistance(u, i)
        print(f"U = {u} V, I = {i} A  ->  R = {r:.1f} Ohm")
    except ZeroDivisionError:
        print(f"U = {u} V, I = {i} A  ->  kein Strom, Widerstand nicht bestimmbar")
    except TypeError as e:
        print(f"U = {u!r}  ->  ungültige Daten ({e})")

U = 12.0 V, I = 0.5 A  ->  R = 24.0 Ohm
U = 5.0 V, I = 0.0 A  ->  kein Strom, Widerstand nicht bestimmbar
U = '230'  ->  ungültige Daten (unsupported operand type(s) for /: 'str' and 'float')


### `else` und `finally` {.unnumbered}

Zwei optionale Blöcke ergänzen `try`/`except`: `else` wird nur ausgeführt, wenn im `try`-Block **kein** Fehler aufgetreten ist; `finally` wird **immer** ausgeführt, typischerweise für Aufräumarbeiten wie das Abschalten eines Geräts. Beim Arbeiten mit Dateien übernimmt das die `with`-Anweisung aus Anhang A, die man einem eigenen `finally` vorziehen sollte.

## Fehler selbst auslösen mit `raise`

Bisher haben wir auf Fehler reagiert, die Python ausgelöst hat. Mindestens ebenso wichtig ist es, in **eigenen Funktionen** Fehler auszulösen, wenn diese mit unsinnigen Werten aufgerufen werden. Dafür gibt es die Anweisung `raise`.

Betrachten wir die Biegespannung in einem Balken mit Rechteckquerschnitt (Breite $b$, Höhe $h$): Mit dem Flächenträgheitsmoment $I = b h^3 / 12$ und dem Randfaserabstand $y = h/2$ gilt $\sigma = M y / I = 6 M / (b h^2)$.

In [6]:
def bending_stress_rect(moment_Nmm, width_mm, height_mm):
    """Biegespannung in N/mm² (MPa) für einen Rechteckquerschnitt."""
    if width_mm <= 0 or height_mm <= 0:
        raise ValueError(f"Breite und Höhe müssen positiv sein (b={width_mm}, h={height_mm}).")
    return 6 * moment_Nmm / (width_mm * height_mm**2)

print(f"{bending_stress_rect(2.0e6, 40, 80):.1f} MPa")

46.9 MPa


In [7]:
bending_stress_rect(2.0e6, 40, -80)  # FEHLER: absichtlich ausgelöst

ValueError: Breite und Höhe müssen positiv sein (b=40, h=-80).

Warum nicht einfach `print("Fehler")` und `return None`? Weil die aufrufende Stelle dann mit `None` weiterrechnet und der Fehler erst viel später – an ganz anderer Stelle – als rätselhafter `TypeError` auftaucht. Eine Exception stoppt das Programm dort, wo das Problem erkannt wurde, beschreibt es mit einer klaren Meldung und lässt dem Aufrufer die Wahl, sie mit `try`/`except` zu behandeln. Wählen Sie einen passenden Fehlertyp: `ValueError` für unzulässige Werte, `TypeError` für falsche Typen.

## Typische Fehler bei der Fehlerbehandlung

::: {.callout-important}
### Fehler nicht verschlucken {.unnumbered}

Die folgende Konstruktion sieht man leider häufig:

```python
try:
    result = compute(data)
except:
    pass
```

Ein `except` ohne Fehlertyp fängt **alle** Fehler ab – auch Tippfehler im eigenen Code (`NameError`) und sogar den Programmabbruch per Strg+C. Zusammen mit `pass` verschwindet jede Spur des Problems, und `result` existiert anschließend womöglich gar nicht. Deshalb:

- Fangen Sie nur Fehlertypen ab, die Sie erwarten und sinnvoll behandeln können (`except ValueError:`).
- Halten Sie den `try`-Block so klein wie möglich, damit Sie nicht versehentlich Fehler aus ganz anderen Zeilen abfangen.
- Wenn Sie einen Fehler abfangen, tun Sie etwas Sinnvolles: einen Ersatzwert verwenden, die Eingabe erneut abfragen oder zumindest eine aussagekräftige Meldung ausgeben.
:::

<div class="exercises">
<b>Übungen:</b>

1. Schreiben Sie eine Funktion `safe_divide(a, b)`, die `a / b` zurückgibt und bei einer Division durch null `float("inf")` liefert.
2. Ergänzen Sie die Funktion `bending_stress_rect` so, dass bei einem Biegemoment, das kein `int` oder `float` ist, ein `TypeError` ausgelöst wird. Rufen Sie die Funktion anschließend in einer Schleife mit den Querschnitten `[(40, 80), (30, 0), (20, 60)]` auf und fangen Sie den Fehler so ab, dass für jeden Querschnitt entweder die Spannung oder eine Fehlermeldung ausgegeben wird. Testen Sie den `TypeError` zusätzlich mit einem Biegemoment als String, z.B. `"2e6"`.
3. Schreiben Sie eine Funktion `read_values(path)`, die eine Datei mit einer Zahl pro Zeile einliest und eine Liste von floats zurückgibt. Existiert die Datei nicht, soll eine leere Liste zurückgegeben und eine Meldung ausgegeben werden; ungültige Zeilen sollen übersprungen werden.

</div>

In [ ]:
#| code-fold: true
#| output: false
# Übung 1
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return float("inf")

print(safe_divide(1, 4), safe_divide(1, 0))

# Übung 2
def bending_stress_rect(moment_Nmm, width_mm, height_mm):
    """Biegespannung in N/mm² (MPa) für einen Rechteckquerschnitt."""
    if not isinstance(moment_Nmm, (int, float)):
        raise TypeError(f"Biegemoment muss eine Zahl sein, nicht {type(moment_Nmm).__name__}.")
    if width_mm <= 0 or height_mm <= 0:
        raise ValueError(f"Breite und Höhe müssen positiv sein (b={width_mm}, h={height_mm}).")
    return 6 * moment_Nmm / (width_mm * height_mm**2)

for b, h in [(40, 80), (30, 0), (20, 60)]:
    try:
        print(f"b={b}, h={h}: {bending_stress_rect(2.0e6, b, h):.1f} MPa")
    except (ValueError, TypeError) as e:
        print(f"b={b}, h={h}: {e}")

try:
    bending_stress_rect("2e6", 40, 80)  # Biegemoment als String
except TypeError as e:
    print("Fehler:", e)

# Übung 3
def read_values(path):
    values = []
    try:
        with open(path, encoding="utf-8") as f:
            for line in f:
                try:
                    values.append(float(line.strip().replace(",", ".")))
                except ValueError:
                    pass  # ungültige Zeile bewusst überspringen (z.B. Kopfzeile)
    except FileNotFoundError:
        print(f"Datei {path} nicht gefunden.")
    return values

print(read_values("data/gibtsnicht.txt"))

0.25 inf
b=40, h=80: 46.9 MPa
b=30, h=0: Breite und Höhe müssen positiv sein (b=30, h=0).
b=20, h=60: 166.7 MPa
Fehler: Biegemoment muss eine Zahl sein, nicht str.
Datei data/gibtsnicht.txt nicht gefunden.
[]
